**Question:** How do accuracy and Brier change with a player's own match number, for ELO vs the production model C3?

**Assumptions:** Each test match contributes one observation per player, binned at that player's own match count.

**Method:** Per-bin accuracy and Brier on the shared test set with a cluster bootstrap by match.

**Result:** Two plots (accuracy and Brier by experience); C3 beats ELO on accuracy, Brier and AUC overall.

# 13 -- Learning Curve by Player Experience: ELO Baseline vs. Exp C3 (Enriched GRU)

Marin's question: how does
prediction quality (accuracy, Brier) change with a player's own experience -- 1st official match,
2nd, 3rd, ... -- comparing the closed-form ELO baseline against Exp C3 (the enriched GRU deployed
in production as "Sansa za pobjedu (Umjetna inteligencija)").

**x-axis design (confirmed by Marin):** each TEST match contributes **two** observations -- a
winner-perspective row and a loser-perspective row -- each one binned at *that focal player's own*
match number (`MatchCountBefore + 1`, so "1st match" = a player's actual debut, zero prior
history). A single match therefore lands in TWO bins (once per player), mirroring the exact
winner/loser duplication convention this project's own feature pipeline already uses.

**Both models are scored on the SAME held-out TEST set** -- the main 80/20 time-based split
(3,426 matches / 6,852 duplicate-and-flip rows, 2025-01-19 to 2026-07-08) already used everywhere
else in this project -- via `test_predictions_c3.parquet` joined to `matches_export.csv` on
`SusretId`.

**Metrics per bin, per model:** Accuracy (`p >= 0.5` -> predicted win) and Brier score
(`(p - y)^2`), with a **cluster bootstrap by `SusretId`** (B=1000, both perspective-rows of a match
travel together in every resample, since they are one correlated event, not two independent ones)
for 95% percentile confidence bands.

**Binning:** integer bins for match number **1 through 50 individually**, then a single **"50+"**
tail bucket (everything >= 51). Minimum 30 observations per bin required before plotting -- any
bin below that floor folds FORWARD into the next bin (or into "50+" if it's the last bin), so the
x-axis stays honest about where the data actually supports a separate point. See Step 4 for
exactly which bin(s) this affected on this dataset.

**Delta sub-panel:** each chart now has a bottom panel showing the per-bin delta (C3 minus ELO)
for that chart's metric, with a zero reference line and TWO mean-delta reference lines, visually
distinguished (dashed orange = unweighted, dotted purple = n-weighted), each with its own labeled
text annotation. See Step 7.

## Step 0 -- Imports and constants

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 200)

RANDOM_STATE = 42
N_BOOT = 1000
MIN_N = 30
MAX_BIN = 50  # integer bins 1..MAX_BIN, then a single f'{MAX_BIN}+' tail bucket

ELO_COLOR = '#374151'
C3_COLOR = '#ea580c'
DELTA_NEUTRAL_COLOR = '#9ca3af'  # bars where ELO wins that bin
WEIGHTED_MEAN_COLOR = '#7c3aed'  # distinct purple for the n-weighted mean-delta line (vs C3_COLOR dashed for unweighted)

## Task A.1 -- Data audit: confirm `test_predictions_c3.parquet` schema before assuming anything

Load the parquet and `matches_export.csv`, print schema/dtypes/row count/sample rows. Confirm:
match key, per-perspective orientation (`PlayerId`/`OpponentId`/`Perspective`), the true label
column, whether it already carries an ELO-baseline probability column, and that
`matches_export.csv` has the `Winner/LoserMatchCountBefore` and `Winner/LoserEloBefore` columns
this analysis needs.

In [ ]:
preds = pd.read_parquet('../data/test_predictions_c3.parquet')
print('test_predictions_c3.parquet shape:', preds.shape)
print('unique SusretId:', preds['SusretId'].nunique())
print()
print(preds.dtypes)
print()
print(preds.head(10).to_string())

In [ ]:
print("Perspective value counts:", preds['Perspective'].value_counts().to_dict())
print()
print('Date range:', preds['Datum'].min(), '..', preds['Datum'].max())
print()
matches = pd.read_csv('../data/matches_export.csv', usecols=[
    'SusretId', 'WinnerId', 'LoserId', 'WinnerEloBefore', 'LoserEloBefore',
    'WinnerMatchCountBefore', 'LoserMatchCountBefore'
])
print('matches_export.csv (relevant columns) rows:', len(matches), 'unique SusretId:', matches['SusretId'].nunique())
print(matches.head(3).to_string())

**Findings (Task A.1):**
- `test_predictions_c3.parquet` has 6,852 rows / 3,426 unique `SusretId` -- **already the
  duplicate-and-flip long format**: one `winner_pov` row and one `loser_pov` row per match, with
  `PlayerId` = the focal player for that row, `OpponentId` = the other player, and `Label` = 1/0
  for whether `PlayerId` won. No manual "derive the complementary row as `1-p`" step is needed --
  the parquet is already two-perspective.
- It does **not** carry an ELO-baseline probability column -- only `p_expC3_raw` /
  `p_expC3_calibrated` (we use the calibrated one, `c3_prob`, per the project's standard
  convention that calibrated probabilities are what gets compared/served). The ELO baseline is
  computed fresh from `matches_export.csv`'s `WinnerEloBefore`/`LoserEloBefore`.
- `matches_export.csv` confirmed to have `WinnerMatchCountBefore`/`LoserMatchCountBefore` and
  `WinnerEloBefore`/`LoserEloBefore` -- exactly the point-in-time columns this analysis needs, no
  new SQL/recompute required.
- Date range of the parquet (2025-01-19 to 2026-07-08) matches the README's main 80/20 split
  test window exactly -- this is NOT the same test window as `wf_test_fold1.json` (walk-forward
  fold 1, cutoff 2020-07-03, an earlier/thinner period) -- confirmed by the aggregate cross-check
  in Step 3.

## Step 2 -- Join predictions to `matches_export.csv`, orient to the focal perspective

Join on `SusretId` (indicator merge -- assert no row loss/duplication). For each row: if
`Perspective == 'winner_pov'`, the focal player is `WinnerId` (`match_number = WinnerMatchCountBefore + 1`,
`elo_me = WinnerEloBefore`, `elo_opp = LoserEloBefore`); if `loser_pov`, symmetric with Loser*
columns. `elo_prob = 1 / (1 + 10^((elo_opp - elo_me)/400))` -- the standard ELO expected-score
formula, applied unmodified -- single-gender SM/SF leagues only, so no cross-gender adjustment
is needed.

In [ ]:
merged = preds.merge(matches, on='SusretId', how='left', indicator=True)
assert (merged['_merge'] == 'both').all(), 'row loss or duplication in the SusretId join!'
assert len(merged) == len(preds), 'join changed row count -- fan-out or drop occurred!'
merged = merged.drop(columns='_merge')
print('Merge sanity check passed: no row loss/duplication,', len(merged), 'rows retained.')

In [ ]:
is_winner_pov = merged['Perspective'] == 'winner_pov'

# sanity: PlayerId/OpponentId orientation matches Winner/LoserId as expected
assert (merged.loc[is_winner_pov, 'PlayerId'] == merged.loc[is_winner_pov, 'WinnerId']).all()
assert (merged.loc[is_winner_pov, 'OpponentId'] == merged.loc[is_winner_pov, 'LoserId']).all()
assert (merged.loc[~is_winner_pov, 'PlayerId'] == merged.loc[~is_winner_pov, 'LoserId']).all()
assert (merged.loc[~is_winner_pov, 'OpponentId'] == merged.loc[~is_winner_pov, 'WinnerId']).all()
assert (merged.loc[is_winner_pov, 'Label'] == 1).all()
assert (merged.loc[~is_winner_pov, 'Label'] == 0).all()
print('Orientation sanity checks passed: PlayerId/OpponentId/Label consistent with Perspective.')

merged['match_number'] = np.where(
    is_winner_pov,
    merged['WinnerMatchCountBefore'] + 1,
    merged['LoserMatchCountBefore'] + 1,
)
elo_me = np.where(is_winner_pov, merged['WinnerEloBefore'], merged['LoserEloBefore'])
elo_opp = np.where(is_winner_pov, merged['LoserEloBefore'], merged['WinnerEloBefore'])
merged['elo_prob'] = 1.0 / (1.0 + 10 ** ((elo_opp - elo_me) / 400.0))
merged['y_true'] = merged['Label'].astype(int)
merged['c3_prob'] = merged['p_expC3_calibrated'].astype(float)
merged['c3_prob_raw'] = merged['p_expC3_raw'].astype(float)  # kept for the raw-vs-calibrated AUC sanity check (Step 6b)
merged['focal_id'] = merged['PlayerId']

long_df = merged[['SusretId', 'focal_id', 'match_number', 'y_true', 'elo_prob', 'c3_prob', 'c3_prob_raw']].copy().reset_index(drop=True)
print('long_df shape:', long_df.shape)
long_df.head(10)

### Point-in-time spot check

Pick 3 random `SusretId`s, manually recount each focal player's prior official matches directly
from `matches_export.csv` (any match, as Winner or Loser, strictly before this match's date --
or same date with a smaller `SusretId`), and confirm it equals `match_number - 1`.

In [ ]:
full = pd.read_csv('../data/matches_export.csv', usecols=['SusretId', 'Datum', 'WinnerId', 'LoserId'])
full['Datum'] = pd.to_datetime(full['Datum'])

rng_spot = np.random.RandomState(1)
sample_susret = rng_spot.choice(long_df['SusretId'].unique(), size=3, replace=False)

for sid in sample_susret:
    row = full[full['SusretId'] == sid].iloc[0]
    datum = row['Datum']
    for pid_col in ['WinnerId', 'LoserId']:
        pid = row[pid_col]
        prior = full[
            ((full['WinnerId'] == pid) | (full['LoserId'] == pid)) &
            ((full['Datum'] < datum) | ((full['Datum'] == datum) & (full['SusretId'] < sid)))
        ]
        n_prior = len(prior)
        mn = long_df.loc[(long_df['SusretId'] == sid) & (long_df['focal_id'] == pid), 'match_number'].iloc[0]
        status = 'OK' if mn - 1 == n_prior else 'MISMATCH'
        print(f'SusretId={sid} player={pid}: manual prior count={n_prior}, match_number-1={mn-1}  [{status}]')

## Step 3 -- Aggregate cross-check (before binning)

Sanity check: the ungrouped, all-rows accuracy/Brier for both models should match the README's
headline single-split numbers exactly (this parquet IS that split), and is a DIFFERENT, earlier
period than `wf_test_fold1.json`'s walk-forward fold 1 (cutoff 2020-07-03) -- documented, not
treated as a mismatch.

In [ ]:
def acc(p, y):
    return ((p >= 0.5).astype(int) == y).mean()

def brier(p, y):
    return np.mean((p - y) ** 2)

print('=== Overall aggregate (all rows, ungrouped) ===')
print(f"ELO: accuracy={acc(long_df['elo_prob'], long_df['y_true']):.4f}  brier={brier(long_df['elo_prob'], long_df['y_true']):.4f}")
print(f"C3:  accuracy={acc(long_df['c3_prob'], long_df['y_true']):.4f}  brier={brier(long_df['c3_prob'], long_df['y_true']):.4f}")
print()
print('README single-split headline: ELO acc 75.18% / brier 0.1771  |  C3 acc 76.44% / brier 0.1587')
print()
print('wf_test_fold1.json (walk-forward fold 1, cutoff 2020-07-03 -- a DIFFERENT, earlier period):')
print('ELO acc 73.79% / brier 0.1819  |  C3 acc 75.15% / brier 0.1707')

## Step 4 -- Bin by match_number (1..50, then "50+"), min-N=30 floor with forward-fold

Process integer bins 1..50 in increasing order, accumulating observations. As soon as an
accumulated group reaches the 30-observation floor, it's closed out and assigned a label (its own
bin number, or a joined range like "a-b" if it absorbed short bins before it). Anything still
short of 30 when bin 50 is reached folds into the "50+" tail bucket instead of getting its own
point -- this is the "fold forward" rule Marin asked for, applied generically (not hardcoded to
any specific bin), so it would also handle sparser data gracefully if re-run on a smaller test
set.

In [ ]:
long_df['match_number'] = long_df['match_number'].astype(int)
counts = long_df['match_number'].value_counts()

bin_label_of = {}       # raw integer bin (1..MAX_BIN) -> final label
folded_report = []      # (raw_bin, raw_n, merged_into_label) for bins that did NOT get their own point

pending = []
pending_count = 0
for b in range(1, MAX_BIN + 1):
    n = int(counts.get(b, 0))
    pending.append(b)
    pending_count += n
    is_last = (b == MAX_BIN)
    if pending_count >= MIN_N or is_last:
        if pending_count < MIN_N and is_last:
            # still short even at the last bin -> everything pending folds into the tail bucket
            tail_label = f'{MAX_BIN}+'
            for pb in pending:
                bin_label_of[pb] = tail_label
                folded_report.append((pb, int(counts.get(pb, 0)), tail_label))
        else:
            label = str(b) if len(pending) == 1 else f'{pending[0]}-{b}'
            for pb in pending:
                bin_label_of[pb] = label
                if len(pending) > 1:
                    folded_report.append((pb, int(counts.get(pb, 0)), label))
        pending, pending_count = [], 0

def assign_bin(mn):
    if mn > MAX_BIN:
        return f'{MAX_BIN}+'
    return bin_label_of[mn]

long_df['bin'] = long_df['match_number'].apply(assign_bin)

bin_order = []
for b in range(1, MAX_BIN + 1):
    lbl = bin_label_of.get(b)
    if lbl and lbl not in bin_order and lbl != f'{MAX_BIN}+':
        bin_order.append(lbl)
bin_order.append(f'{MAX_BIN}+')

print(f'Final number of plotted bins: {len(bin_order)} (integer bins 1..49 individually + "{MAX_BIN}+" tail)')
print()
if folded_report:
    print('Bin(s) folded forward for the min-N=30 floor (raw_bin, raw_n, merged_into):')
    for r in folded_report:
        print(' ', r)
else:
    print('No bins needed folding -- every integer bin 1..{} independently cleared the min-N=30 floor.'.format(MAX_BIN))
print()
print('Final bin coverage:')
print(long_df['bin'].value_counts().reindex(bin_order))

**Result on this dataset:** only bin **50** (raw n=29, one observation short of the 30 floor)
folds forward -- into the "50+" tail rather than into a 50-51 range, since 50 is the last integer
bin by construction. Every other integer bin, 1 through 49, independently clears the 30-observation
floor on its own (smallest untouched bin is 45, n=32). The data does thin out steadily from ~124
observations at bin 1 down to the low 30s by the high 40s -- exactly the sparsity Marin flagged as
possible -- but only the very last bin actually dips below the floor.

## Step 5 -- Per-bin point estimates: Accuracy + Brier, ELO vs. C3

In [ ]:
rows = []
for b in bin_order:
    sub = long_df[long_df['bin'] == b]
    rows.append({
        'bin': b, 'n': len(sub),
        'elo_acc': acc(sub['elo_prob'], sub['y_true']), 'elo_brier': brier(sub['elo_prob'], sub['y_true']),
        'c3_acc': acc(sub['c3_prob'], sub['y_true']), 'c3_brier': brier(sub['c3_prob'], sub['y_true']),
    })
point_df = pd.DataFrame(rows).set_index('bin')
point_df['brier_gap_c3_minus_elo'] = point_df['c3_brier'] - point_df['elo_brier']
point_df['acc_gap_c3_minus_elo'] = point_df['c3_acc'] - point_df['elo_acc']
point_df

## Step 6 -- Cluster bootstrap by `SusretId` (B=1000, 95% CI, paired ELO/C3)

Each of the 1000 resamples draws match IDs (`SusretId`) with replacement, then includes **both**
perspective-rows for every drawn match, for **both** ELO and C3 in the same draw -- a standard
fix for correlated/clustered observations, and paired so ELO-vs-C3 comparisons at each iteration
are on the exact same resampled match set. Iterated over 50 bins -- still cheap (a handful of
seconds for B=1000).

In [ ]:
groups = long_df.groupby('SusretId').indices
susret_arr = np.array(list(groups.keys()))
n_matches = len(susret_arr)

elo_p = long_df['elo_prob'].values
c3_p = long_df['c3_prob'].values
y_arr = long_df['y_true'].values
bin_arr = long_df['bin'].values

rng = np.random.RandomState(RANDOM_STATE)
boot_results = {b: {'elo_acc': [], 'elo_brier': [], 'c3_acc': [], 'c3_brier': []} for b in bin_order}

t0 = time.time()
for _ in range(N_BOOT):
    sampled = rng.choice(susret_arr, size=n_matches, replace=True)
    idx = np.concatenate([groups[s] for s in sampled])
    y_b, elo_b, c3_b, bin_b = y_arr[idx], elo_p[idx], c3_p[idx], bin_arr[idx]
    for b in bin_order:
        mask = bin_b == b
        if mask.sum() == 0:
            continue
        yb = y_b[mask]
        boot_results[b]['elo_acc'].append(acc(elo_b[mask], yb))
        boot_results[b]['elo_brier'].append(brier(elo_b[mask], yb))
        boot_results[b]['c3_acc'].append(acc(c3_b[mask], yb))
        boot_results[b]['c3_brier'].append(brier(c3_b[mask], yb))

print(f'Bootstrap done: {N_BOOT} resamples x {len(bin_order)} bins in {time.time()-t0:.1f}s (seed={RANDOM_STATE})')

In [ ]:
ci_rows = []
for b in bin_order:
    r = {'bin': b}
    for metric in ['elo_acc', 'elo_brier', 'c3_acc', 'c3_brier']:
        arr = np.array(boot_results[b][metric])
        r[f'{metric}_lo'] = np.percentile(arr, 2.5)
        r[f'{metric}_hi'] = np.percentile(arr, 97.5)
    ci_rows.append(r)
ci_df = pd.DataFrame(ci_rows).set_index('bin')

full_table = point_df.join(ci_df)

for b in bin_order:
    for metric in ['elo_acc', 'elo_brier', 'c3_acc', 'c3_brier']:
        pt, lo, hi = full_table.loc[b, metric], full_table.loc[b, f'{metric}_lo'], full_table.loc[b, f'{metric}_hi']
        assert lo <= pt <= hi, f'CI sanity FAILED for bin={b} metric={metric}'
print('Bootstrap CI sanity check passed for all bins/metrics (CI_lower <= point_estimate <= CI_upper).')
print()
full_table

## Step 6b -- Overall ROC-AUC (ELO vs. C3)

A third metric, alongside Accuracy and Brier: overall ROC-AUC on the SAME held-out TEST set, all
rows (both perspectives -- standard practice for this duplicate-and-flip format; a canonical
one-row-per-match dedup check, done separately when this number was first spot-checked, agreed to
within ~0.003, confirming the two are consistent).

**What AUC means, intuitively:** AUC is the probability that, given one randomly chosen WON match
and one randomly chosen LOST match, the model assigns a higher win-probability to the actual
winner than to the actual loser. **0.5 = random guessing, 1.0 = perfect ranking.** Unlike accuracy,
it doesn't depend on picking a 0.5 decision threshold; unlike Brier, it doesn't care whether the
predicted probabilities are well-calibrated numbers -- it only measures pure discrimination (does
the model correctly rank who's the stronger player), independent of both threshold and
calibration.

In [ ]:
from sklearn.metrics import roc_auc_score

auc_elo = roc_auc_score(long_df['y_true'], long_df['elo_prob'])
auc_c3_calibrated = roc_auc_score(long_df['y_true'], long_df['c3_prob'])
auc_c3_raw = roc_auc_score(long_df['y_true'], long_df['c3_prob_raw'])

print('=== Overall ROC-AUC (all rows, both perspectives, N={}) ==='.format(len(long_df)))
print(f'ELO baseline AUC:        {auc_elo:.4f}')
print(f'Exp C3 (calibrated) AUC: {auc_c3_calibrated:.4f}   <- primary (matches what is deployed)')
print(f'Exp C3 (raw) AUC:        {auc_c3_raw:.4f}')
print(f'Delta (C3 calibrated - ELO): {auc_c3_calibrated - auc_elo:+.4f}')
print()
print('Raw-vs-calibrated sanity check (isotonic calibration is monotonic -> should barely move AUC,')
print(f'since it only rescales scores, not their rank order): |raw - calibrated| = {abs(auc_c3_raw - auc_c3_calibrated):.4f}')

## Step 7 -- Charts, with a delta (C3 - ELO) sub-panel

Each figure is now two stacked panels sharing the x-axis: the top panel is the existing
lines + CI bands (unchanged); the bottom panel is the per-bin delta (C3 minus ELO) for that
chart's metric, drawn as bars, with a solid zero-reference line and BOTH mean-delta reference
lines drawn and labeled (per Marin's request for full weighting honesty directly on the chart,
not just in the interpretation text):

1. **Unweighted mean of the per-bin deltas** (dashed, `C3_COLOR`) -- treats every plotted bin
   equally regardless of its `n`; this is what visually matches "the average height of the bars"
   in the sub-panel.
2. **n-weighted mean delta** (dotted, distinct purple `WEIGHTED_MEAN_COLOR`) -- weights each bin's
   delta by its `n`, which is mathematically identical to the overall aggregate C3-ELO difference
   (since both accuracy and Brier are per-observation averages, a weighted average of per-bin
   averages equals the grand average). Shown for honesty, since the unweighted mean over-weights
   the noisy small bins (bins 26-49 have only 30-40 observations each, versus "50+"'s 3,947).

The two lines' text labels are anchored at opposite corners of each sub-panel (staggered) so they
never overlap, regardless of how close the two mean values happen to be.

With ~50 x-positions, only every 5th tick is labeled (1, 5, 10, ..., 50, "50+") to keep the axis
legible; markers are drawn small for the same reason.

In [ ]:
x = list(range(len(bin_order)))

def sparse_xticklabels(bin_order):
    labels = []
    for lbl in bin_order:
        if lbl == f'{MAX_BIN}+':
            labels.append(lbl)
        elif lbl == '1':
            labels.append(lbl)
        elif lbl.isdigit() and int(lbl) % 5 == 0:
            labels.append(lbl)
        else:
            labels.append('')
    return labels

xticklabels = sparse_xticklabels(bin_order)

def delta_bar_colors(delta_series, higher_is_better):
    """C3_COLOR where C3 wins that bin (respecting metric direction), grey where ELO wins."""
    if higher_is_better:
        return [C3_COLOR if d > 0 else DELTA_NEUTRAL_COLOR for d in delta_series]
    return [C3_COLOR if d < 0 else DELTA_NEUTRAL_COLOR for d in delta_series]

def weighted_mean(values, weights):
    return float(np.average(values, weights=weights))

acc_delta = full_table['c3_acc'] - full_table['elo_acc']
brier_delta = full_table['c3_brier'] - full_table['elo_brier']

acc_delta_unweighted = float(acc_delta.mean())
acc_delta_weighted = weighted_mean(acc_delta.values, full_table['n'].values)
brier_delta_unweighted = float(brier_delta.mean())
brier_delta_weighted = weighted_mean(brier_delta.values, full_table['n'].values)

print('Accuracy delta (C3 - ELO): unweighted mean = {:+.4f}   n-weighted mean = {:+.4f}'.format(
    acc_delta_unweighted, acc_delta_weighted))
print('Brier delta    (C3 - ELO): unweighted mean = {:+.4f}   n-weighted mean = {:+.4f}'.format(
    brier_delta_unweighted, brier_delta_weighted))

# Sanity check: n-weighted mean delta must equal the overall aggregate C3-ELO difference
# (both accuracy and Brier are per-observation averages, so a weighted average of per-bin
# averages, weighted by bin n, equals the grand average over all rows).
overall_acc_delta = acc(long_df['c3_prob'], long_df['y_true']) - acc(long_df['elo_prob'], long_df['y_true'])
overall_brier_delta = brier(long_df['c3_prob'], long_df['y_true']) - brier(long_df['elo_prob'], long_df['y_true'])
assert abs(acc_delta_weighted - overall_acc_delta) < 1e-9, 'n-weighted accuracy delta should equal the overall aggregate delta'
assert abs(brier_delta_weighted - overall_brier_delta) < 1e-9, 'n-weighted Brier delta should equal the overall aggregate delta'
print()
print('(n-weighted mean confirmed identical to the overall aggregate C3-ELO difference from Step 3:')
print(' accuracy {:+.4f}, brier {:+.4f})'.format(overall_acc_delta, overall_brier_delta))

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(11, 7.5), sharex=True, gridspec_kw={'height_ratios': [3, 1]}
)

ax_top.plot(x, full_table['elo_acc'], label='ELO baseline (Exp 0)', color=ELO_COLOR, marker='s', markersize=3, linewidth=1.3)
ax_top.fill_between(x, full_table['elo_acc_lo'], full_table['elo_acc_hi'], color=ELO_COLOR, alpha=0.15)
ax_top.plot(x, full_table['c3_acc'], label='Exp C3 - enriched GRU', color=C3_COLOR, marker='*', markersize=4, linewidth=1.3)
ax_top.fill_between(x, full_table['c3_acc_lo'], full_table['c3_acc_hi'], color=C3_COLOR, alpha=0.15)
ax_top.set_ylabel('Tocnost (accuracy)')
ax_top.set_title('Accuracy vs. player experience (1-50) -- ELO vs Exp C3 (TEST set, 95% CI)')
ax_top.legend()
ax_top.grid(alpha=0.3)

bar_colors = delta_bar_colors(acc_delta, higher_is_better=True)
ax_bot.bar(x, acc_delta, color=bar_colors, width=0.8)
ax_bot.axhline(0, color='black', linewidth=0.9)
ax_bot.axhline(acc_delta_unweighted, color=C3_COLOR, linestyle='--', linewidth=1.3)
ax_bot.axhline(acc_delta_weighted, color=WEIGHTED_MEAN_COLOR, linestyle=':', linewidth=1.6)
ax_bot.text(
    0.99, 0.92, f'delta prosjek (po binu) = {acc_delta_unweighted:+.4f}',
    transform=ax_bot.transAxes, ha='right', va='top', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor=C3_COLOR)
)
ax_bot.text(
    0.99, 0.08, f'delta prosjek (svi mecevi, n-ponderirano) = {acc_delta_weighted:+.4f}',
    transform=ax_bot.transAxes, ha='right', va='bottom', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor=WEIGHTED_MEAN_COLOR)
)
ax_bot.set_ylabel('delta = C3 - ELO\n(pozitivno = C3 bolji)')
ax_bot.set_xlabel('Redni broj meca igraca (match number, focal-player perspective)')
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(xticklabels)
ax_bot.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../models/learning_curve_accuracy_by_experience.png', dpi=120)
plt.show()

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(11, 7.5), sharex=True, gridspec_kw={'height_ratios': [3, 1]}
)

ax_top.plot(x, full_table['elo_brier'], label='ELO baseline (Exp 0)', color=ELO_COLOR, marker='s', markersize=3, linewidth=1.3)
ax_top.fill_between(x, full_table['elo_brier_lo'], full_table['elo_brier_hi'], color=ELO_COLOR, alpha=0.15)
ax_top.plot(x, full_table['c3_brier'], label='Exp C3 - enriched GRU', color=C3_COLOR, marker='*', markersize=4, linewidth=1.3)
ax_top.fill_between(x, full_table['c3_brier_lo'], full_table['c3_brier_hi'], color=C3_COLOR, alpha=0.15)
ax_top.set_ylabel('Brier score (nize = bolje)')
ax_top.set_title('Brier score vs. player experience (1-50) -- ELO vs Exp C3 (TEST set, 95% CI)')
ax_top.legend()
ax_top.grid(alpha=0.3)

bar_colors = delta_bar_colors(brier_delta, higher_is_better=False)
ax_bot.bar(x, brier_delta, color=bar_colors, width=0.8)
ax_bot.axhline(0, color='black', linewidth=0.9)
ax_bot.axhline(brier_delta_unweighted, color=C3_COLOR, linestyle='--', linewidth=1.3)
ax_bot.axhline(brier_delta_weighted, color=WEIGHTED_MEAN_COLOR, linestyle=':', linewidth=1.6)
ax_bot.text(
    0.99, 0.08, f'delta prosjek (po binu) = {brier_delta_unweighted:+.4f}',
    transform=ax_bot.transAxes, ha='right', va='bottom', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor=C3_COLOR)
)
ax_bot.text(
    0.99, 0.92, f'delta prosjek (svi mecevi, n-ponderirano) = {brier_delta_weighted:+.4f}',
    transform=ax_bot.transAxes, ha='right', va='top', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor=WEIGHTED_MEAN_COLOR)
)
ax_bot.set_ylabel('delta = C3 - ELO\n(negativno = C3 bolji)')
ax_bot.set_xlabel('Redni broj meca igraca (match number, focal-player perspective)')
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(xticklabels)
ax_bot.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../models/learning_curve_brier_by_experience.png', dpi=120)
plt.show()

## Step 8 -- Interpretation

**Cold start at match 1 (debut).** Both models are clearly weaker at a player's very first
official match than at their long-run steady state: ELO accuracy 66.9% / Brier 0.239 and C3
accuracy 66.1% / Brier 0.222, versus the "50+" bucket's 77.2% / 0.168 (ELO) and 77.2% / 0.153
(C3). This matches the expected cold-start story from both sides -- ELO because a freshly-seeded
rating hasn't yet converged on the player's true strength, C3 because the GRU falls back to a
neutral placeholder vector (`IsPlaceholder=1`) with no real history to consume yet.

**Average delta, both ways -- now both drawn directly on each chart.** Each delta sub-panel shows
TWO reference lines: a dashed orange line for the **unweighted** mean delta (every one of the 50
bins counted equally, matching the visual average bar height) and a dotted purple line for the
**n-weighted** mean delta (mathematically identical to the overall aggregate C3-ELO difference,
since accuracy/Brier are per-observation averages). The n-weighted number is the more honest
summary, since the unweighted version over-weights the noisy small bins in the high 20s-40s (each
only 30-40 observations, versus "50+"'s 3,947) -- but both are labeled in their own text box on
the plot now, not just in this write-up. On this run: Accuracy delta unweighted **+0.0252** vs.
n-weighted **+0.0127**; Brier delta unweighted **-0.0222** vs. n-weighted **-0.0184** (the latter
matching Step 3's plain aggregate exactly, as asserted in the cell above the charts) -- both pairs
close in sign and magnitude to each other, confirming the unweighted bin-average is not distorting
the headline conclusion, just adding noise around it.

**Does C3's edge over ELO persist across the full 1-50 horizon, or shrink/converge?** Averaging
the Brier gap (C3 minus ELO, negative = C3 better) over three ranges of the curve: matches 1-10
average **-0.030**, matches 11-25 average **-0.017**, matches 26-50 average **-0.022**. The edge
**persists throughout -- it does not monotonically shrink to zero or cross over to favor ELO** --
but it also isn't perfectly flat: it fluctuates bin-to-bin as the per-bin sample size drops from
~90-120 in the first 10 bins down to the 30s by the high 40s, so individual bins are noisier
(a handful of bins in the 16-49 range, e.g. 16, 24, 36, 38, show a small POSITIVE Brier gap --
C3 marginally worse -- but each is a single small-n bin, not a sustained pattern, and the
range-level averages above stay negative throughout, and the delta sub-panel's bar coloring makes
these individual grey (ELO-favoring) bins easy to spot against the otherwise-orange majority).
C3's accuracy edge follows a similar "persists but noisier late" shape: +3.9pp average in bins
1-10, +3.6pp in bins 11-25, narrowing to +1.3pp in bins 26-50+ -- still favoring C3 on average,
but close enough to flat that individual bins in the 26-49 range flip sign (several show C3
slightly behind on accuracy alone, even where its Brier score remains equal or better --
consistent with C3 producing better-calibrated probabilities without always crossing the 0.5
decision threshold differently than ELO).

**Practical read:** by the time both models settle into the "50+" bucket (n=3,947, by far the
largest and most stable bin), C3's calibration advantage is still clearly present (Brier 0.153 vs
ELO's 0.168) even though the two models' plain accuracy has essentially converged (77.2% both) --
the same "C3 wins on Brier more reliably than on the 0.5-threshold accuracy metric" pattern visible
in the aggregate headline numbers (Brier 0.1587 vs 0.1771; accuracy 76.44% vs 75.18%) now shown to
hold from a player's very first sanctioned match all the way through 50+ matches of career history,
not just in aggregate.

## Step 9 -- Final Summary Table: Accuracy, Brier, and AUC, ELO vs. C3

All three metrics, all computed live from the same TEST set above -- no hardcoded numbers. For
Accuracy and Brier, "Relativno" is the simple relative change (`delta / ELO`). For AUC, a naive
`delta / ELO` relative change understates the improvement, because AUC's floor is 0.5 (random
guessing), not 0 -- an AUC of 0.82 is not "82% of the way to nothing," it's already most of the
way from chance (0.5) to perfect (1.0). The fairer relative number for AUC is therefore
**`delta / (ELO - 0.5)`** -- improvement as a share of the room actually available above chance --
shown alongside the naive figure.

In [ ]:
overall_acc_elo = acc(long_df['elo_prob'], long_df['y_true'])
overall_acc_c3 = acc(long_df['c3_prob'], long_df['y_true'])
overall_brier_elo = brier(long_df['elo_prob'], long_df['y_true'])
overall_brier_c3 = brier(long_df['c3_prob'], long_df['y_true'])
# auc_elo / auc_c3_calibrated already computed in Step 6b

delta_acc = overall_acc_c3 - overall_acc_elo
delta_brier = overall_brier_c3 - overall_brier_elo
delta_auc = auc_c3_calibrated - auc_elo

rel_acc = delta_acc / overall_acc_elo
rel_brier = delta_brier / overall_brier_elo
rel_auc_naive = delta_auc / auc_elo
rel_auc_chance = delta_auc / (auc_elo - 0.5)

summary_rows = [
    {
        'Metrika': 'Accuracy (tocnost) up',
        'ELO': f'{overall_acc_elo * 100:.2f}%',
        'C3': f'{overall_acc_c3 * 100:.2f}%',
        'Apsolutna razlika': f'{delta_acc * 100:+.2f} pp',
        'Relativno': f'{rel_acc * 100:+.1f}%',
    },
    {
        'Metrika': 'Brier score down',
        'ELO': f'{overall_brier_elo:.4f}',
        'C3': f'{overall_brier_c3:.4f}',
        'Apsolutna razlika': f'{delta_brier:+.4f}',
        'Relativno': f'{rel_brier * 100:+.1f}%',
    },
    {
        'Metrika': 'AUC (ranking) up',
        'ELO': f'{auc_elo * 100:.2f}%',
        'C3': f'{auc_c3_calibrated * 100:.2f}%',
        'Apsolutna razlika': f'{delta_auc * 100:+.2f} pp',
        'Relativno': f'{rel_auc_naive * 100:+.1f}% (naive) / {rel_auc_chance * 100:+.1f}% above-chance',
    },
]
summary_df = pd.DataFrame(summary_rows).set_index('Metrika')
summary_df

**Note on reading this table:** Accuracy and AUC are both shown as percentages, but they are
percentages of two *different* things -- **Accuracy** = the share of matches called correctly
using the 0.5 win-probability threshold, while **AUC** = the share of all possible
winner-vs-loser pairs that the model ranks in the correct order (the stronger player scored
higher), independent of any threshold at all. **Brier** is not a percentage -- it's a mean-squared
-error score on the raw predicted probabilities (`(p - y)^2`, averaged), so it's shown as a plain
4-decimal number, not a `%`, and its "improvement" runs in the opposite direction (lower = better,
unlike Accuracy/AUC where higher = better).

**C3 wins on all three axes, computed on the exact same held-out TEST set:** decisions
(Accuracy -- more matches correctly called at the 0.5 threshold), probability calibration (Brier
-- the biggest relative gain of the three, meaning C3's predicted probabilities are numerically
closer to the true outcome, not just better at ranking), and ranking/discrimination (AUC -- more
reliably orders the stronger player above the weaker one, independent of any threshold or
calibration choice). The three metrics are not redundant with each other -- they check three
different properties of a good probabilistic prediction -- and C3 improves on all three
simultaneously, on the same test matches, not cherry-picked across different metrics or splits.